# 12 — Indicadores do painel (Power BI)
**TCC — Preços e concentração de fornecedores em compras públicas de insumos hospitalares, 2009–2023**

**Objetivo.** Gerar, a partir da MESMA base usada no texto, todas as tabelas que o painel do Power BI lê. Todas as contas
com regra são feitas aqui, em Python; o Power BI só agrega o que está filtrado (D13, D24).

**Fase 2 (out/2026).** A rotina lê a **base analítica** (rotina 04B, D14): item = CATMAT + unidade (D17), preço e valor em
reais de dez/2023 (D20), régua de valores implausíveis sobre o preço real (D16), só as classes de saúde (D41), classe CATMAT
oficial (D40). Filtrar no painel é olhar um pedaço dessa base, nunca outra.

**Indicadores (descritivos, no painel e na seção "Painéis construídos" do texto):**
1. diferença de preço entre compradores do mesmo item no mesmo ano (P90/P10; item-ano com 10 compras ou mais, D44);
2. preço por modalidade de compra, comparado ao pregão no mesmo item, ano e faixa de lote (3 compras ou mais, D45);
3. época da compra (mês informado pela instituição como data da compra — Metadados BPS, MS, abr/2026);
4. repetição do fornecedor pelo mesmo comprador no mesmo item, em anos seguidos (os "fornecedores estreantes" saíram, D52);
5. quem domina o item: fatia do maior fornecedor pelo valor e pelo número de compras, item-ano com 7 compras ou mais (D28, D29);
6. página do item: quanto cada estado paga (3 compras ou mais, D45/D46) e a série do preço do item (10 compras ou mais, D44/D46);
7. concentração por recorte (D25): por item-ano, por item-ano-estado e por item-ano-comprador, sempre com 7 compras ou mais (D7).

Os testes que sustentam cada indicador estão na rotina 11 (seções 13 e 14).

**Entradas:** `Base Analitica`, `Base Classificacao Receita`, saídas das rotinas 05 a 08. **Saída:** pasta `Base Painel`.

**Alteração de 21/10/2026 (complemento da D24):** a `dim_comprador` ganha a coluna `comprador_rotulo` (CNPJ + razão social). Se existir a pasta `Base Painel anterior` (cópia da Base Painel feita antes de rodar), a última célula compara as duas pelo conteúdo.

In [1]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass   # fora do Colab: use a variável de ambiente BPS_DADOS
PASTA_DADOS = os.environ.get('BPS_DADOS', '/content/drive/MyDrive/TCC/Registro de Compras v2/Base v2')
# ─────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


In [2]:
!pip install polars==1.35.2 --quiet

In [3]:
from pathlib import Path
import polars as pl
pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(60); pl.Config.set_tbl_width_chars(200)

BASE     = Path(PASTA_DADOS)
ARQ_BASE = BASE / 'Base Analitica' / 'base_analitica.parquet'                    # rotina 04B (D14)
P_REC    = BASE / 'Base Classificacao Receita' / 'compradores_receita.parquet'
SAIDAS   = {'indice': BASE / 'Base Preco Real - Publico' / 'indice_pareado.csv',
            'robustez': BASE / 'Base Preco Real - Publico' / 'robustez_indice.csv',
            'hhi': BASE / 'Base HHI - Publico' / 'hhi_concentracao.parquet',
            'conc_ano': BASE / 'Base HHI - Publico' / 'concentracao_por_ano.csv',
            'territorial': BASE / 'Base Territorial - Publico' / 'indice_territorial_uf.csv',
            'territorial_ano': BASE / 'Base Territorial - Publico' / 'indice_territorial_uf_ano.csv',   # D43
            'territorial_periodo': BASE / 'Base Territorial - Publico' / 'indice_territorial_uf_periodo.csv',   # D62
            'escala': BASE / 'Base Escala - Publico' / 'escala_quintis.csv'}
SAIDA    = BASE / 'Base Painel'
SAIDA_ANTERIOR = BASE / 'Base Painel anterior'   # opcional: cópia da Base Painel antes de rodar, só para a conferência da última célula
SAIDA.mkdir(parents=True, exist_ok=True)
ANOS = range(2009, 2024)
for p in [ARQ_BASE, P_REC, *SAIDAS.values()]:
    print(f'{"OK " if p.exists() else "FALTANDO"}  {p}')
assert all(p.exists() for p in [ARQ_BASE, P_REC, *SAIDAS.values()]), 'Rode as rotinas 04B a 08 antes, e confira PASTA_DADOS'

# parâmetros (cada um com a sua decisão no Registro de decisões)
MIN_COMPRAS    = 7    # D7: mínimo de compras no item-ano para medir concentração (HHI e fatia do maior fornecedor, D28)
MIN_DISPERSAO  = 10   # D44: P10 e P90 exigem 10 compras (com menos, 10% é menos de uma compra; Auditoria 13f)
MIN_MODALIDADE = 3    # D45: mediana com 3 preços ou mais (IN SEGES/ME 65/2021, art. 6º), o mesmo mínimo da régua
MIN_ESTADO_ITEM = 3   # D46/D45: estado na página do item com 3 compras ou mais do item no ano
LIM_NAO, LIM_ALTO = 1500, 2500   # D7: faixas do CADE (2016)
QUINTIS = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']   # D31
CORTE_ITEM_GRANDE = 0.8   # D29: item grande = compras iguais ou acima do corte do quintil superior (percentil 80) do ano

GRAVAR_CSV_FATO = False   # a tabela central em CSV passa de 200 MB; o Power BI lê o Parquet diretamente

def salvar(df, nome, csv=True):
    df.write_parquet(SAIDA / f'{nome}.parquet')
    if csv: df.write_csv(SAIDA / f'{nome}.csv')
    print(f'  -> Base Painel/{nome}  ({df.height:,} linhas)')

OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Analitica/base_analitica.parquet
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Classificacao Receita/compradores_receita.parquet
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Preco Real - Publico/indice_pareado.csv
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Preco Real - Publico/robustez_indice.csv
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base HHI - Publico/hhi_concentracao.parquet
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base HHI - Publico/concentracao_por_ano.csv
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Territorial - Publico/indice_territorial_uf.csv
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Territorial - Publico/indice_territorial_uf_ano.csv
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Territorial - Publico/indice_territorial_uf_pe

## 1. Tabela central de compras (`fato_compras`)
A própria base analítica, no menor grão (cada compra; D13). `valor_implausivel` vem da rotina 04B (D16): a compra marcada
sai das contas de preço e de valor, mas fica nas contagens. `compras_item_ano` conta todas as compras do item no ano;
`compras_validas_item_ano` conta as que entram em preço (base dos mínimos D44, D45 e D46).

In [4]:
fato = pl.read_parquet(ARQ_BASE).with_columns(pl.col('valor_implausivel').fill_null(False))
valida = ~pl.col('valor_implausivel')
FATO_COLS = ['id_compra', 'data_compra', 'ano', 'mes', 'item', 'cod_catmat', 'unidade_chave', 'unidade_marca_generico',
             'classe_catmat', 'cnpj_instituicao', 'uf', 'municipio', 'local_recuperado', 'origem_local', 'cnpj_fornecedor',
             'modalidade_grupo', 'modalidade_compra', 'preco_unitario', 'quantidade', 'valor_total', 'fator_deflacao',
             'preco_real', 'valor_real', 'valor_implausivel', 'compras_item_ano', 'compras_validas_item_ano',
             'mediana_item_ano', 'preco_rel_mediana', 'faixa_lote', 'lote_rel']
fato_out = fato.select([c for c in FATO_COLS if c in fato.columns])
# D61: as regras do Python gravadas por extenso em cada compra, para o Power BI só filtrar "Sim" (nenhuma regra no DAX)
fato_out = fato_out.with_columns(
    pl.when(~pl.col('valor_implausivel')).then(pl.lit('Sim')).otherwise(pl.lit('Não')).alias('compra_valida'),   # D16
    pl.when(~pl.col('valor_implausivel') & (pl.col('compras_validas_item_ano').fill_null(0) >= MIN_MODALIDADE))   # D16 + D45 (3)
      .then(pl.lit('Sim')).otherwise(pl.lit('Não')).alias('entra_preco_relativo'))
print(f'Tabela central do painel (base analítica): {fato_out.height:,} compras')
print(f'  com valor implausível (D16)             : {int(fato_out["valor_implausivel"].sum()):,}')
print(f'  válidas para preço                      : {fato.filter(valida).height:,}')
print(f'  D61 compra_valida = Sim                 : {(fato_out["compra_valida"] == "Sim").sum():,}')
print(f'  D61 entra_preco_relativo = Sim          : {(fato_out["entra_preco_relativo"] == "Sim").sum():,}')
salvar(fato_out, 'fato_compras', csv=GRAVAR_CSV_FATO)

Tabela central do painel (base analítica): 833,928 compras
  com valor implausível (D16)             : 31,268
  válidas para preço                      : 802,660
  D61 compra_valida = Sim                 : 802,660
  D61 entra_preco_relativo = Sim          : 747,684
  -> Base Painel/fato_compras  (833,928 linhas)


## 2. Tabelas de apoio (dimensões)

In [5]:
# Item: um rótulo e uma descrição por item; classe CATMAT oficial com o nome do catálogo (D40), sem agrupamento do trabalho
moda = lambda c: pl.col(c).drop_nulls().mode().sort().first()
_limpo = lambda c: pl.col(c).str.replace_all(r'\s+', ' ').str.strip_chars()   # o catálogo traz espaços duplos
nome_frase = lambda c: (_limpo(c).str.slice(0, 1).str.to_uppercase() + _limpo(c).str.slice(1).str.to_lowercase())
dim_item = (fato.sort(['data_compra', 'id_compra']).group_by('item')
               .agg(pl.col('cod_catmat').first(), pl.col('unidade_chave').first(), moda('unidade_rotulo').alias('unidade_rotulo'),
                    pl.col('desc_item').drop_nulls().last().alias('descricao'), pl.col('classe_catmat').first(),
                    pl.col('nome_classe').first(), pl.len().alias('compras'),
                    pl.col('ano').min().alias('primeiro_ano'), pl.col('ano').max().alias('ultimo_ano'))
               .with_columns((pl.col('classe_catmat') + ' — ' + nome_frase('nome_classe')).alias('classe_rotulo'),   # filtro "Classe CATMAT" (D24, D40)
                             (pl.col('cod_catmat') + ' — ' + pl.col('descricao').str.slice(0, 80)).alias('produto_rotulo'))   # produto = código + descrição (D24)
               .sort(['compras', 'item'], descending=[True, False]))
assert dim_item.select(pl.col('item').is_unique().all()).item()
print(dim_item.group_by('classe_rotulo').agg(pl.len().alias('itens'), pl.col('compras').sum()).sort('compras', descending=True))
salvar(dim_item, 'dim_item')

# Comprador: nome e natureza jurídica da Receita Federal (rotina 03C); esfera derivada da natureza jurídica
rec = pl.read_parquet(P_REC).select(pl.col('cnpj').alias('cnpj_instituicao'), 'razao_social', 'natureza_cod', 'natureza_desc', 'ente_federativo')
d = pl.col('natureza_desc')
loc = (fato.group_by('cnpj_instituicao').agg(moda('uf').alias('uf'), moda('municipio').alias('municipio'), pl.len().alias('compras'),
                                             pl.col('ano').min().alias('primeiro_ano'), pl.col('ano').max().alias('ultimo_ano')))
dim_comprador = (loc.join(rec, on='cnpj_instituicao', how='left')
                    .with_columns(pl.when(d.str.contains('Consórcio')).then(pl.lit('Consórcio público'))
                                    .when(d.is_in(['Empresa Pública', 'Sociedade de Economia Mista'])).then(pl.lit('Empresa estatal'))
                                    .when(d.str.contains('Federal') & ~d.str.contains('Estadual')).then(pl.lit('Federal'))
                                    .when(d.str.contains('Estadual')).then(pl.lit('Estadual'))
                                    .when(d.str.contains('Munic')).then(pl.lit('Municipal'))
                                    .otherwise(pl.lit('Outro')).alias('esfera'),
                                  pl.when(d.str.contains('Fundo')).then(pl.lit('Fundo de saúde'))
                                    .when(d.is_in(['Município', 'Órgão Público do Poder Executivo Municipal'])).then(pl.lit('Prefeitura e órgãos municipais'))
                                    .when(d.str.contains('Consórcio')).then(pl.lit('Consórcio público'))
                                    .when(d.str.contains('Fundação|Autarquia')).then(pl.lit('Fundação ou autarquia'))
                                    .when(d.str.contains('Órgão Público')).then(pl.lit('Secretaria ou órgão estadual/federal'))
                                    .otherwise(pl.lit('Empresa estatal')).alias('tipo_comprador'))
                    .sort(['compras', 'cnpj_instituicao'], descending=[True, False]))
assert dim_comprador['razao_social'].null_count() == 0, 'comprador sem cadastro da Receita'
print(dim_comprador.group_by('esfera').agg(pl.len().alias('compradores'), pl.col('compras').sum()).sort('compras', descending=True))
# Complemento da D24 (21/10/2026, decisão da autora): comprador no filtro = CNPJ + razão social,
# porque a razão social se repete (ex.: "FUNDO MUNICIPAL DE SAUDE" = 291 CNPJs). Mesmo padrão do produto_rotulo.
_c = pl.col('cnpj_instituicao')
dim_comprador = dim_comprador.with_columns(
    (_c.str.slice(0, 2) + '.' + _c.str.slice(2, 3) + '.' + _c.str.slice(5, 3) + '/' + _c.str.slice(8, 4) + '-' + _c.str.slice(12, 2)
     + ' — ' + pl.col('razao_social')).alias('comprador_rotulo'))
assert dim_comprador['cnpj_instituicao'].str.len_chars().eq(14).all(), 'CNPJ fora do padrão de 14 dígitos'
assert dim_comprador['comprador_rotulo'].is_unique().all(), 'rótulo de comprador repetido'
print(f'comprador_rotulo: {dim_comprador.height:,} compradores, {dim_comprador["comprador_rotulo"].n_unique():,} rótulos únicos')
print(dim_comprador.filter(pl.col('cnpj_instituicao') == '48468284000171')['comprador_rotulo'].to_list())
salvar(dim_comprador, 'dim_comprador')

# Fornecedor: um nome por CNPJ (o mais recente na fonte) e anos com venda. Sem "ano de estreia" (D52): a base começa em
# 2009 e é de registro voluntário, então o primeiro ano na base não é a entrada do fornecedor no mercado
dim_fornecedor = (fato.sort(['data_compra', 'id_compra']).group_by('cnpj_fornecedor')
                      .agg(pl.col('fornecedor').drop_nulls().last().alias('nome'), pl.col('fornecedor').drop_nulls().n_unique().alias('grafias_na_fonte'),
                           pl.len().alias('compras'), pl.col('ano').max().alias('ultimo_ano'),
                           pl.col('ano').n_unique().alias('anos_com_venda'))
                      .sort(['compras', 'cnpj_fornecedor'], descending=[True, False]))
print(f'Fornecedores: {dim_fornecedor.height:,} | com mais de uma grafia de nome na fonte: {(dim_fornecedor["grafias_na_fonte"] > 1).sum():,}')
salvar(dim_fornecedor, 'dim_fornecedor')

# Modalidade: grupos e fundamento legal (rotina 03, dicionário GRUPO_MODALIDADE)
base_legal = {'Pregão': 'Lei 10.520/2002', 'Concorrência': 'Lei 8.666/1993, art. 22, I', 'Tomada de preços': 'Lei 8.666/1993, art. 22, II',
              'Convite': 'Lei 8.666/1993, art. 22, III', 'Dispensa': 'Lei 8.666/1993, art. 24 (inclui emergência, inciso IV)',
              'Inexigibilidade': 'Lei 8.666/1993, art. 25', 'Registro de preços': 'Lei 8.666/1993, art. 15 (procedimento, não modalidade)',
              'Outras': 'rótulos da fonte sem correspondência inequívoca na lei'}
dim_modalidade = (fato.group_by('modalidade_grupo').agg(pl.len().alias('compras'), pl.col('modalidade_compra').unique().sort().str.join(' | ').alias('rotulos_da_fonte'))
                     .with_columns(pl.col('modalidade_grupo').replace_strict(base_legal, default=None).alias('base_legal'))
                     .sort('compras', descending=True))
print(dim_modalidade)
salvar(dim_modalidade, 'dim_modalidade')

MESES = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
# Fase 2: em vez de marcar anos como "base pequena" (sem decisão registrada), cada ano traz o seu número de compras (D32);
# a única ressalva escrita é a de 2023 (D21)
compras_ano = fato.group_by('ano').agg(pl.len().alias('compras_no_ano'))
dim_calendario = (pl.DataFrame({'ano': [a for a in ANOS for _ in range(12)], 'mes': [m for _ in ANOS for m in range(1, 13)]})
                    .with_columns(pl.col('ano').cast(pl.Int32), pl.col('mes').cast(pl.Int8))
                    .with_columns(pl.col('mes').map_elements(lambda m: MESES[m - 1], return_dtype=pl.Utf8).alias('nome_mes'),
                                  pl.when(pl.col('mes') <= 6).then(1).otherwise(2).alias('semestre'),
                                  pl.when(pl.col('ano') == 2023).then(pl.lit('volume baixo de abril a dezembro (D21)')).otherwise(pl.lit('')).alias('ressalva'))
                    .join(compras_ano, on='ano', how='left'))
salvar(dim_calendario, 'dim_calendario')

shape: (16, 3)
┌───────────────────────────────────────────────────────────────┬───────┬─────────┐
│ classe_rotulo                                                 ┆ itens ┆ compras │
│ ---                                                           ┆ ---   ┆ ---     │
│ str                                                           ┆ u32   ┆ u32     │
╞═══════════════════════════════════════════════════════════════╪═══════╪═════════╡
│ 6505 — Drogas e medicamentos                                  ┆ 7091  ┆ 681653  │
│ 6515 — Instrumentos, equipamentos e suprimentos médicos e ci… ┆ 6148  ┆ 76384   │
│ 6520 — Instrumentos, equipamentos e suprimentos dentários     ┆ 1967  ┆ 25359   │
│ 6532 — Vestuário hospitalar e cirúrgico e itens correlatos d… ┆ 495   ┆ 14086   │
│ 6510 — Materiais cirúrgicos para curativos                    ┆ 859   ┆ 12314   │
│ 6550 — Substâncias para diagnóstico "in vitro", reagentes, c… ┆ 666   ┆ 7098    │
│ 6810 — Produtos químicos                                   

## 3. Indicador 1 — diferença de preço entre compradores (D44)
Para cada item-ano com pelo menos 10 compras válidas (D44; sem valores implausíveis, D16): percentis do preço real e razão
P90/P10 (quanto paga quem está entre os 10% mais caros em relação a quem está entre os 10% mais baratos). A versão
"controlando o lote" mede a mesma razão depois de retirar o efeito da faixa de quantidade. Esta tabela também alimenta a série
"Preço do item ao longo do tempo" da página do item (mediana e faixa P10–P90, D46).
Respaldo: Bandiera, Prat e Valletti (2009); Rossignoli, Pontarolo e Fernandez-Llimos (2024).

In [6]:
v = fato.filter(valida & (pl.col('compras_validas_item_ano') >= MIN_DISPERSAO))   # D44, D16
v = v.with_columns((pl.col('preco_real').log() - pl.col('mediana_item_ano').log()).alias('dp'))
v = v.with_columns((pl.col('dp') - pl.col('dp').median().over('faixa_lote')).alias('dp_sem_lote'))
q = lambda c, x: pl.col(c).quantile(x, 'linear')
disp = (v.group_by('item', 'ano')
          .agg(pl.len().alias('compras'), pl.col('cnpj_instituicao').n_unique().alias('compradores'),
               pl.col('preco_real').median().alias('mediana'), q('preco_real', .10).alias('p10'), q('preco_real', .25).alias('p25'),
               q('preco_real', .75).alias('p75'), q('preco_real', .90).alias('p90'),
               (q('dp_sem_lote', .90) - q('dp_sem_lote', .10)).exp().alias('p90_p10_sem_lote'))
          .with_columns((pl.col('p90') / pl.col('p10')).alias('p90_p10'), (pl.col('p75') / pl.col('p25')).alias('p75_p25'))
          .sort('item', 'ano'))
print(f'Itens-ano com >= {MIN_DISPERSAO} compras: {disp.height:,}')
print(f'  mediana P90/P10: {disp["p90_p10"].median():.2f} | mediana P75/P25: {disp["p75_p25"].median():.2f} '
      f'| P90/P10 controlando o lote: {disp["p90_p10_sem_lote"].median():.2f}')
print(f'  itens-ano em que P90 passa do dobro de P10: {(disp["p90_p10"] > 2).mean():.1%}')
salvar(disp, 'ind_dispersao_item_ano')

Itens-ano com >= 10 compras: 14,188
  mediana P90/P10: 2.40 | mediana P75/P25: 1.51 | P90/P10 controlando o lote: 2.34
  itens-ano em que P90 passa do dobro de P10: 68.4%
  -> Base Painel/ind_dispersao_item_ano  (14,188 linhas)


## 4. Indicador 2 — preço por modalidade de compra (D45)
Desvio do preço em relação à mediana do item no ano, por grupo de modalidade, de três formas: (a) bruto;
(b) comparado ao pregão no MESMO item, ano e faixa de lote (a medida publicada); (c) número de células comparadas.
D57: as células da comparação (b) também vão para o painel (`ind_modalidade_celulas`, uma linha por modalidade, item, ano e
faixa de lote). O Power BI tira a mediana das células que ficam no filtro (período, classe, produto, unidade) e mostra o
número de células ao lado; sem filtro, reproduz a `ind_modalidade`. Estado e comprador não entram (D57).
Itens-ano com pelo menos 3 compras válidas (D45), sem valores implausíveis.
Respaldo: Lei 8.666/1993 (arts. 22, 24 e 25), Lei 10.520/2002; Coviello, Guglielmo e Spagnolo (2018).

In [7]:
w = fato.filter(valida & (pl.col('compras_validas_item_ano') >= MIN_MODALIDADE))   # D45, D16
w = w.with_columns((pl.col('preco_real').log() - pl.col('mediana_item_ano').log()).alias('dp'))
pct = lambda e: ((e.exp() - 1) * 100).round(1)
bruto = w.group_by('modalidade_grupo').agg(pl.len().alias('compras'), pct(pl.col('dp').median()).alias('desvio_bruto_pct'),
                                           pl.col('lote_rel').median().round(2).alias('lote_mediano_rel'))
pg = (w.filter(pl.col('modalidade_grupo') == 'Pregão').group_by('item', 'ano', 'faixa_lote')
       .agg(pl.col('dp').median().alias('dp_pg'), pl.len().alias('compras_pregao')))
# D57: as células (modalidade x item x ano x faixa de lote) ficam gravadas para o painel
ind_mod_cel = (w.filter(pl.col('modalidade_grupo') != 'Pregão').group_by('modalidade_grupo', 'item', 'ano', 'faixa_lote')
                .agg(pl.col('dp').median().alias('dp_m'), pl.len().alias('compras_modalidade'))
                .join(pg, on=['item', 'ano', 'faixa_lote'])
                .with_columns((pl.col('dp_m') - pl.col('dp_pg')).alias('dif_log_vs_pregao'))
                .select('modalidade_grupo', 'item', 'ano', 'faixa_lote', 'compras_modalidade', 'compras_pregao', 'dif_log_vs_pregao')
                .sort('modalidade_grupo', 'item', 'ano', 'faixa_lote'))
par = (ind_mod_cel.group_by('modalidade_grupo')
         .agg(pl.len().alias('celulas_comparadas'), pct(pl.col('dif_log_vs_pregao').median()).alias('vs_pregao_mesmo_lote_pct')))
ind_mod = bruto.join(par, on='modalidade_grupo', how='left').sort('compras', descending=True)
print(ind_mod)
salvar(ind_mod, 'ind_modalidade')
# D57: no Power BI, % vs pregão = (EXP(MEDIAN(dif_log_vs_pregao)) - 1) * 100; células = número de linhas no filtro
print(f'Células da comparação por modalidade (D57): {ind_mod_cel.height:,}')
salvar(ind_mod_cel, 'ind_modalidade_celulas')

shape: (8, 6)
┌────────────────────┬─────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────────────┐
│ modalidade_grupo   ┆ compras ┆ desvio_bruto_pct ┆ lote_mediano_rel ┆ celulas_comparadas ┆ vs_pregao_mesmo_lote_pct │
│ ---                ┆ ---     ┆ ---              ┆ ---              ┆ ---                ┆ ---                      │
│ str                ┆ u32     ┆ f64              ┆ f64              ┆ u32                ┆ f64                      │
╞════════════════════╪═════════╪══════════════════╪══════════════════╪════════════════════╪══════════════════════════╡
│ Pregão             ┆ 672849  ┆ 0.0              ┆ 1.0              ┆ null               ┆ null                     │
│ Dispensa           ┆ 38894   ┆ 10.9             ┆ 0.22             ┆ 14548              ┆ 7.4                      │
│ Tomada de preços   ┆ 12984   ┆ 12.3             ┆ 0.4              ┆ 7647               ┆ 10.3                     │
│ Registro de preços ┆ 9776    ┆ -

## 5. Indicador 3 — época da compra
Compras por mês, segundo a data da compra informada pela instituição compradora (Metadados BPS, MS, abr/2026).
Contraste com a literatura: Liebman e Mahoney (2017) observam concentração no fim do ano fiscal nos EUA.
Todos os anos, de 2009 a 2023, cada um com o seu número de compras (D51). Em 2023 o volume de abril a dezembro é muito
baixo (nova versão do BPS; informe do Ministério da Saúde, D21), o que infla a participação do 1º semestre.

In [8]:
# D27: sem soma em reais no painel; a época da compra é medida pelo número de compras
saz = (fato.group_by('ano', 'mes').agg(pl.len().alias('compras'))
          .with_columns((pl.col('compras') / pl.col('compras').sum().over('ano') * 100).round(2).alias('pct_compras_ano'))
          .sort('ano', 'mes'))
# D51: todos os anos (2009–2023), cada um com o seu número de compras; 2023 com a ressalva da D21
s1 = (saz.group_by('ano').agg(pl.col('compras').sum().alias('compras_no_ano'),
                              pl.col('pct_compras_ano').filter(pl.col('mes') <= 6).sum().round(1).alias('pct_1o_semestre')).sort('ano'))
print('Compras no 1º semestre, por ano (D51):'); print(s1)
salvar(saz, 'ind_sazonalidade')

Compras no 1º semestre, por ano (D51):
shape: (15, 3)
┌──────┬────────────────┬─────────────────┐
│ ano  ┆ compras_no_ano ┆ pct_1o_semestre │
│ ---  ┆ ---            ┆ ---             │
│ i32  ┆ u32            ┆ f64             │
╞══════╪════════════════╪═════════════════╡
│ 2009 ┆ 5105           ┆ 38.0            │
│ 2010 ┆ 10063          ┆ 47.9            │
│ 2011 ┆ 8299           ┆ 71.8            │
│ 2012 ┆ 3214           ┆ 69.5            │
│ 2013 ┆ 2169           ┆ 59.3            │
│ 2014 ┆ 19384          ┆ 47.3            │
│ 2015 ┆ 97587          ┆ 57.3            │
│ 2016 ┆ 89188          ┆ 67.6            │
│ 2017 ┆ 86018          ┆ 70.5            │
│ 2018 ┆ 146193         ┆ 59.5            │
│ 2019 ┆ 129354         ┆ 54.3            │
│ 2020 ┆ 71014          ┆ 63.3            │
│ 2021 ┆ 70125          ┆ 60.4            │
│ 2022 ┆ 68433          ┆ 63.5            │
│ 2023 ┆ 27782          ┆ 83.7            │
└──────┴────────────────┴─────────────────┘
  -> Base Painel/ind_s

## 6. Indicador 4 — repetição do fornecedor (D52)
Para cada comprador que comprou o mesmo item em dois anos seguidos: algum fornecedor do ano anterior venceu de novo?
Respaldo: Coviello, Guglielmo e Spagnolo (2018). Duas ressalvas: (1) repetir não é favorecimento, o fornecedor pode ter o
melhor preço de novo; (2) uma mesma ata de registro de preços pode gerar compras em dois anos civis seguidos (Lei 8.666/1993,
art. 15, § 3º, III; Decreto 7.892/2013, art. 12; Lei 14.133/2021, art. 84), e a base não identifica a ata.
Os "fornecedores estreantes" foram avaliados e descartados (D52): a base começa em 2009, os arquivos de 2000–2008 não têm
CNPJ, e o registro é voluntário, então o primeiro ano de um fornecedor na base mede a adesão dos compradores ao BPS,
não a entrada no mercado.

In [9]:
fy = fato.group_by('ano').agg(pl.col('cnpj_fornecedor').n_unique().alias('fornecedores'))   # D52: só a contagem, sem estreantes
ci = fato.group_by('cnpj_instituicao', 'item', 'ano').agg(pl.col('cnpj_fornecedor').unique().alias('fs'))
rep = (ci.join(ci.with_columns((pl.col('ano') + 1).alias('ano')), on=['cnpj_instituicao', 'item', 'ano'], suffix='_ant')
         .with_columns(pl.col('fs').list.set_intersection(pl.col('fs_ant')).list.len().gt(0).alias('repete'))
         .group_by('ano').agg(pl.len().alias('pares_comprador_item'), (pl.col('repete').mean() * 100).round(1).alias('pct_repete_fornecedor')))
ind_forn = fy.join(rep, on='ano', how='left').sort('ano')
tot_rep = ci.join(ci.with_columns((pl.col('ano') + 1).alias('ano')), on=['cnpj_instituicao', 'item', 'ano'], suffix='_ant') \
            .select(pl.col('fs').list.set_intersection(pl.col('fs_ant')).list.len().gt(0).mean() * 100).item()
print(ind_forn)
print(f'Período inteiro: algum fornecedor do ano anterior vence de novo em {tot_rep:.1f}% dos pares comprador-item')
salvar(ind_forn, 'ind_fornecedores_ano')

shape: (15, 4)
┌──────┬──────────────┬──────────────────────┬───────────────────────┐
│ ano  ┆ fornecedores ┆ pares_comprador_item ┆ pct_repete_fornecedor │
│ ---  ┆ ---          ┆ ---                  ┆ ---                   │
│ i32  ┆ u32          ┆ u32                  ┆ f64                   │
╞══════╪══════════════╪══════════════════════╪═══════════════════════╡
│ 2009 ┆ 327          ┆ null                 ┆ null                  │
│ 2010 ┆ 440          ┆ 1424                 ┆ 40.6                  │
│ 2011 ┆ 401          ┆ 1159                 ┆ 41.0                  │
│ 2012 ┆ 251          ┆ 823                  ┆ 39.7                  │
│ 2013 ┆ 222          ┆ 614                  ┆ 37.6                  │
│ 2014 ┆ 537          ┆ 571                  ┆ 37.1                  │
│ 2015 ┆ 1078         ┆ 5553                 ┆ 35.5                  │
│ 2016 ┆ 981          ┆ 17751                ┆ 41.3                  │
│ 2017 ┆ 1068         ┆ 11910                ┆ 32.9           

## 7. Indicador 5 — quem domina o item: o mapa de oportunidades (D28, D29)
Cada ponto do mapa é um **item em um ano** com 7 compras ou mais naquele ano (o mesmo recorte do HHI, D7); o ano vem do
filtro de período. Duas medidas lado a lado (D28): a fatia do maior fornecedor pelo **valor** (medida principal, compras
válidas, em reais de dez/2023) e pelo **número de compras**. A fatia é mostrada como número contínuo, de 0% a 100%, com a linha
de 20% como referência legal (D29: Lei 12.529/2011, art. 36, §2º, como analogia declarada), sem classes.
A fatia pelo valor só é calculada com 7 compras válidas ou mais, a mesma regra do HHI por valor (D54); abaixo disso fica vazia,
e as colunas `compras_valor_valido`, `compras_implausiveis` e `situacao_fatia_valor` mostram o motivo. O ponto continua no mapa. Item grande =
item-ano com compras iguais ou acima do corte do quintil superior do ano (percentil 80), entre os item-ano do mapa (D29, D31). O tamanho do mercado é o número de compras (D27).

In [10]:
LINHA_REFERENCIA_LEGAL = 0.20   # D29: Lei 12.529/2011, art. 36, §2º — só uma linha de referência no gráfico, não classifica
mapa = fato.filter(pl.col('compras_item_ano') >= MIN_COMPRAS)   # D7/D28: recorte por item-ano
def fatia_lider(df, peso, nome):
    w_ = df.group_by('item', 'ano', 'cnpj_fornecedor').agg(peso.alias('w'))
    return w_.group_by('item', 'ano').agg((pl.col('w').max() / pl.col('w').sum()).alias(nome),
                                          pl.col('cnpj_fornecedor').filter(pl.col('w') == pl.col('w').max()).sort().first().alias(f'lider_{nome}'))
dominio = (mapa.group_by('item', 'ano').agg(pl.len().alias('compras'), pl.col('cnpj_fornecedor').n_unique().alias('fornecedores'),
                                            valida.sum().alias('compras_valor_valido'))
               .join(fatia_lider(mapa.filter(valida), pl.col('valor_real').sum(), 'fatia_lider_valor'), on=['item', 'ano'], how='left')
               .join(fatia_lider(mapa, pl.len().cast(pl.Float64), 'fatia_lider_compras'), on=['item', 'ano'], how='left')
               # D54: a fatia POR VALOR só existe com 7 compras válidas ou mais (mesma regra do HHI por valor, D7/D16);
               # abaixo disso fica vazia (nunca zero) e a coluna de situação diz o motivo. O ponto continua no mapa.
               .with_columns((pl.col('compras') - pl.col('compras_valor_valido')).alias('compras_implausiveis'),
                             (pl.col('compras_valor_valido') >= MIN_COMPRAS).alias('_valor_ok'))
               .with_columns(pl.when(pl.col('_valor_ok')).then(pl.col('fatia_lider_valor')).otherwise(None).alias('fatia_lider_valor'),
                             pl.when(pl.col('_valor_ok')).then(pl.col('lider_fatia_lider_valor')).otherwise(None).alias('lider_fatia_lider_valor'),
                             pl.when(pl.col('_valor_ok')).then(pl.lit('Calculada'))
                               .otherwise(pl.lit(f'Não calculada: menos de {MIN_COMPRAS} compras com valor válido')).alias('situacao_fatia_valor'))
               .drop('_valor_ok')
               # D29: item grande = compras IGUAIS OU ACIMA do corte do quintil superior do ano (a mesma regra da rotina 11, seção 14)
               .with_columns(pl.col('compras').quantile(CORTE_ITEM_GRANDE, 'linear').over('ano').alias('corte_item_grande_no_ano'))
               .with_columns((pl.col('compras') >= pl.col('corte_item_grande_no_ano')).alias('item_grande'))
               .join(dim_item.select('item', 'classe_rotulo', 'produto_rotulo', 'unidade_rotulo'), on='item', how='left')
               .sort('ano', 'item'))
print(dominio.group_by('ano').agg(pl.len().alias('itens_no_mapa'), pl.col('item_grande').sum().alias('itens_grandes'),
                                   pl.col('fatia_lider_valor').is_null().sum().alias('sem_fatia_valor_D54'),
                                   (pl.col('fatia_lider_valor').median() * 100).round(1).alias('fatia_mediana_valor_pct'),
                                   (pl.col('fatia_lider_compras').median() * 100).round(1).alias('fatia_mediana_compras_pct')).sort('ano'))
salvar(dominio, 'ind_dominio_item_ano')

shape: (15, 6)
┌──────┬───────────────┬───────────────┬─────────────────────┬─────────────────────────┬───────────────────────────┐
│ ano  ┆ itens_no_mapa ┆ itens_grandes ┆ sem_fatia_valor_D54 ┆ fatia_mediana_valor_pct ┆ fatia_mediana_compras_pct │
│ ---  ┆ ---           ┆ ---           ┆ ---                 ┆ ---                     ┆ ---                       │
│ i32  ┆ u32           ┆ u32           ┆ u32                 ┆ f64                     ┆ f64                       │
╞══════╪═══════════════╪═══════════════╪═════════════════════╪═════════════════════════╪═══════════════════════════╡
│ 2009 ┆ 192           ┆ 56            ┆ 12                  ┆ 60.1                    ┆ 28.6                      │
│ 2010 ┆ 480           ┆ 99            ┆ 3                   ┆ 61.3                    ┆ 30.0                      │
│ 2011 ┆ 382           ┆ 87            ┆ 13                  ┆ 64.0                    ┆ 28.6                      │
│ 2012 ┆ 59            ┆ 17            ┆ 6       

## 8. Indicador 6 — página do item: quanto cada estado paga (D46)
Para cada item, ano e estado: mediana do preço real do estado ÷ mediana do item no ano (escala centrada em 1, D29), com o
número de compras ao lado. Entram estados com 3 compras válidas ou mais do item no ano (D45, D46). A série do preço do item
ao longo do tempo usa a tabela do indicador 1 (anos com 10 compras ou mais, D44).

In [11]:
tem_uf = pl.col('uf').is_not_null() & (pl.col('uf') != '')
item_uf = (fato.filter(valida & tem_uf)
               .group_by('item', 'ano', 'uf').agg(pl.len().alias('compras'), pl.col('preco_real').median().alias('mediana_estado'),
                                                  pl.col('mediana_item_ano').first())
               .filter(pl.col('compras') >= MIN_ESTADO_ITEM)
               .with_columns((pl.col('mediana_estado') / pl.col('mediana_item_ano')).round(3).alias('preco_relativo'))
               .sort('item', 'ano', 'uf'))
print(f'Item-ano-estado com >= {MIN_ESTADO_ITEM} compras: {item_uf.height:,} | preço relativo mediano: {item_uf["preco_relativo"].median():.3f}')
salvar(item_uf, 'ind_item_estado_ano')

Item-ano-estado com >= 3 compras: 65,596 | preço relativo mediano: 1.000
  -> Base Painel/ind_item_estado_ano  (65,596 linhas)


## 9. Indicador 7 — concentração por recorte (D25)
A concentração acompanha todos os filtros do painel. Período, classe, produto e unidade só escolhem item-ano, então usam a
tabela da rotina 06 (`res_hhi`). Estado e comprador mudam o mercado olhado, então o HHI é recalculado aqui para cada
item-ano-estado e cada item-ano-comprador, com as mesmas regras da rotina 06: 7 compras ou mais (D7), "Dados insuficientes"
abaixo disso, faixas do CADE, HHI por valor só com compras válidas (D16) e em reais de dez/2023 (D20).

In [12]:
cnpjf = pl.col('cnpj_fornecedor').cast(pl.Utf8).str.strip_chars()
reg = fato.filter(cnpjf.is_not_null() & (cnpjf.str.len_chars() > 0) & (cnpjf != '00000000000000')
                  & (pl.col('quantidade') > 0) & (pl.col('valor_real') > 0))
def hhi_recorte(df, chave):
    def um(d, peso, suf):
        w_ = pl.len().cast(pl.Float64) if peso == 'compras' else pl.col('valor_real').sum()
        g = (d.group_by(chave + ['cnpj_fornecedor']).agg(w_.alias('w'), pl.len().alias('r'))
              .with_columns((pl.col('w') / pl.col('w').sum().over(chave) * 100).alias('s')))
        return g.group_by(chave).agg((pl.col('s') ** 2).sum().round(6).alias(f'hhi_{suf}'), pl.col('r').sum().alias(f'n_{suf}'))   # 6 casas: ver rotina 06
    def classe(h, n):
        return (pl.when(pl.col(n) < MIN_COMPRAS).then(pl.lit('Dados insuficientes'))
                  .when(pl.col(h) < LIM_NAO).then(pl.lit('Não Concentrado'))
                  .when(pl.col(h) <= LIM_ALTO).then(pl.lit('Moderadamente Concentrado'))
                  .otherwise(pl.lit('Altamente Concentrado')))
    return (um(df, 'compras', 'compras').join(um(df.filter(valida), 'valor', 'valor'), on=chave, how='left')
              .with_columns(classe('hhi_compras', 'n_compras').alias('classe_compras'),
                            pl.when(pl.col('hhi_valor').is_null()).then(pl.lit('Dados insuficientes'))
                              .otherwise(classe('hhi_valor', 'n_valor')).alias('classe_valor')))
def esvaziar_insuficientes(df):
    # D59 (regra geral da D54; D7): onde a classe é "Dados insuficientes" (menos de 7 compras), o HHI e a fatia do líder ficam
    # VAZIOS no painel, nunca zero. A rotina 06 continua guardando o número para diagnóstico; só as tabelas do painel ficam vazias
    ops = []
    for medida in ['compras', 'valor']:
        insuf = pl.col(f'classe_{medida}') == 'Dados insuficientes'
        for c in [f'hhi_{medida}', f'maior_fatia_{medida}']:
            if c in df.columns:
                ops.append(pl.when(insuf).then(None).otherwise(pl.col(c)).alias(c))
    return df.with_columns(ops)

conc_uf = esvaziar_insuficientes(hhi_recorte(reg.filter(pl.col('uf').is_not_null() & (pl.col('uf') != '')), ['ano', 'item', 'uf']))
conc_comp = esvaziar_insuficientes(hhi_recorte(reg, ['ano', 'item', 'cnpj_instituicao']))
for nome, t in [('por estado', conc_uf), ('por comprador', conc_comp)]:
    k = t.filter(pl.col('classe_compras') != 'Dados insuficientes')
    print(f'Concentração {nome}: {t.height:,} combinações | com >= {MIN_COMPRAS} compras: {k.height:,} '
          f'({k.height / t.height:.1%}) | alta por compras entre elas: {(k["classe_compras"] == "Altamente Concentrado").mean():.1%}')
salvar(conc_uf, 'ind_concentracao_item_ano_uf')
salvar(conc_comp, 'ind_concentracao_item_ano_comprador')

Concentração por estado: 260,900 combinações | com >= 7 compras: 25,188 (9.7%) | alta por compras entre elas: 22.4%
Concentração por comprador: 726,451 combinações | com >= 7 compras: 1,108 (0.2%) | alta por compras entre elas: 92.0%
  -> Base Painel/ind_concentracao_item_ano_uf  (260,900 linhas)
  -> Base Painel/ind_concentracao_item_ano_comprador  (726,451 linhas)


## 10. Resultados das rotinas 05 a 08 (copiados sem alteração, exceto o HHI: vazio onde a classe é "Dados insuficientes", D59) e tabela de rastreabilidade

In [13]:
for nome, p in SAIDAS.items():
    df = pl.read_parquet(p) if p.suffix == '.parquet' else pl.read_csv(p)
    if nome == 'hhi':   # D59: no painel, sem número onde a classe é "Dados insuficientes"
        df = esvaziar_insuficientes(df)
        res_hhi_painel = df
    salvar(df, f'res_{nome}')

rastro = pl.DataFrame([
    ('Compras, compradores, itens e fornecedores', '1', '03, 03B, 03D, 04B, 12', 'fato_compras, dim_*', 'base analítica: compras públicas, preço > 0, classes de saúde (D14, D41)', 'Método: base de dados'),
    ('Diferença de preço entre compradores (P90/P10)', '2', '12', 'ind_dispersao_item_ano', f'item-ano com >= {MIN_DISPERSAO} compras (D44); sem implausíveis (D16)', 'Painéis construídos'),
    ('Preço do item: mediana e faixa por ano', '2', '12', 'ind_dispersao_item_ano', f'anos com >= {MIN_DISPERSAO} compras (D44, D46); preço real; sem implausíveis', 'Painéis construídos'),
    ('Quanto cada estado paga (página do item)', '2', '12', 'ind_item_estado_ano', f'estados com >= {MIN_ESTADO_ITEM} compras do item no ano (D45, D46)', 'Painéis construídos'),
    ('Preço por modalidade (vs pregão, mesmo lote)', '3', '03, 12', 'ind_modalidade, ind_modalidade_celulas, dim_modalidade', f'item-ano com >= {MIN_MODALIDADE} compras (D45); grupos pela Lei 8.666/1993 e 10.520/2002; células acompanham período, classe, produto e unidade (D57)', 'Painéis construídos'),
    ('Efeito do tamanho do lote', '3', '08', 'res_escala', 'item = CATMAT + unidade', 'Resultados: escala'),
    ('Diferença de preço entre estados', '3', '07', 'res_territorial, res_territorial_ano, res_territorial_periodo', 'itens-ano com >= 5 UFs; UFs com >= 500 compras no período (D23); por ano e por período contínuo sem corte, com o número de compras (D43, D62)', 'Resultados: território'),
    ('Concentração por compras e por valor (HHI)', '4', '06, 12', 'res_hhi, res_conc_ano, ind_concentracao_item_ano_uf, ind_concentracao_item_ano_comprador', f'>= {MIN_COMPRAS} compras (D7); faixas do CADE; recalculada por estado e por comprador (D25)', 'Resultados: concentração'),
    ('Quem domina o item (mapa de oportunidades)', '4', '12', 'ind_dominio_item_ano', f'item-ano com >= {MIN_COMPRAS} compras; fatia pelo valor e pelas compras (D28); fatia por valor só com >= {MIN_COMPRAS} compras válidas, senão vazia com o motivo (D54); item grande = compras >= percentil 80 do ano (D29)', 'Painéis construídos'),
    ('Repetição do fornecedor', '4', '12', 'ind_fornecedores_ano, dim_fornecedor', 'todas as compras da base analítica (D16: contagens incluem as implausíveis); sem estreantes (D52)', 'Painéis construídos'),
    ('Época da compra', '5', '12', 'ind_sazonalidade', 'data da compra informada pela instituição; todos os anos com o número de compras (D51); 2023 com ressalva (D21)', 'Painéis construídos'),
    ('Índice de preço real 2009-2023', '5', '05', 'res_indice, res_robustez', 'itens pareados CATMAT + unidade; IPCA dez/2023; item-ano com >= 3 compras válidas (D45); a versão sem mínimo fica como teste', 'Resultados: preço real'),
], schema=['indicador', 'pagina_painel', 'rotina', 'arquivo_base_painel', 'regra', 'secao_do_texto'], orient='row')
print(rastro)
salvar(rastro, 'rastreabilidade')

  -> Base Painel/res_indice  (15 linhas)
  -> Base Painel/res_robustez  (15 linhas)
  -> Base Painel/res_hhi  (78,600 linhas)
  -> Base Painel/res_conc_ano  (15 linhas)
  -> Base Painel/res_territorial  (23 linhas)
  -> Base Painel/res_territorial_ano  (288 linhas)
  -> Base Painel/res_territorial_periodo  (2,760 linhas)
  -> Base Painel/res_escala  (5 linhas)
shape: (12, 6)
┌───────────────────────────────────────────┬───────────────┬───────────────────────┬───────────────────────────────────────────┬───────────────────────────────────────────┬──────────────────────────┐
│ indicador                                 ┆ pagina_painel ┆ rotina                ┆ arquivo_base_painel                       ┆ regra                                     ┆ secao_do_texto           │
│ ---                                       ┆ ---           ┆ ---                   ┆ ---                                       ┆ ---                                       ┆ ---                      │
│ str              

## 11. Conferência

In [14]:
print(f'fato_compras            : {fato_out.height:,} compras (base analítica)')
print(f'compradores / fornecedores: {dim_comprador.height:,} / {dim_fornecedor.height:,}')
print(f'itens (CATMAT + unidade)  : {dim_item.height:,}')
assert fato.filter(valida).height == pl.read_csv(BASE / 'Base Preco Real - Publico' / 'serie_preco_anual.csv')['registros'].sum(), \
    'compras válidas divergem da rotina 05'
hhi06 = pl.read_parquet(SAIDAS['hhi'])
assert dominio.height == hhi06.filter(pl.col('n_compras') >= MIN_COMPRAS).height, 'mapa diverge das combinações classificadas da rotina 06'
print('\nValores para conferência (calculados nesta execução):')
print(f'  P90/P10 mediano {disp["p90_p10"].median():.2f} ({disp["p90_p10_sem_lote"].median():.2f} controlando o lote; '
      f'{(disp["p90_p10"] > 2).mean():.1%} acima de 2)')
print('  modalidade vs pregão no mesmo lote: ' + ' | '.join(f'{r["modalidade_grupo"]} {r["vs_pregao_mesmo_lote_pct"]:+.1f}%'
                                                         for r in ind_mod.iter_rows(named=True) if r['vs_pregao_mesmo_lote_pct'] is not None))
print(f'  1º semestre (2009-2023, D51): {s1["pct_1o_semestre"].min()}% a {s1["pct_1o_semestre"].max()}%')
print(f'  itens grandes no mapa em 2023 (D29): {dominio.filter(pl.col("ano") == 2023)["item_grande"].sum()}')
print(f'  fornecedor do ano anterior vence de novo em {tot_rep:.1f}% dos pares comprador-item')
print('\nArquivos gerados:')
for f in sorted(SAIDA.glob('*.parquet')): print(' ', f.name)

fato_compras            : 833,928 compras (base analítica)
compradores / fornecedores: 2,027 / 4,703
itens (CATMAT + unidade)  : 19,979

Valores para conferência (calculados nesta execução):
  P90/P10 mediano 2.40 (2.34 controlando o lote; 68.4% acima de 2)
  modalidade vs pregão no mesmo lote: Dispensa +7.4% | Tomada de preços +10.3% | Registro de preços -3.0% | Concorrência -0.1% | Convite +53.2% | Outras +17.6% | Inexigibilidade +7.1%
  1º semestre (2009-2023, D51): 38.0% a 83.7%
  itens grandes no mapa em 2023 (D29): 158
  fornecedor do ano anterior vence de novo em 30.9% dos pares comprador-item

Arquivos gerados:
  dim_calendario.parquet
  dim_comprador.parquet
  dim_fornecedor.parquet
  dim_item.parquet
  dim_modalidade.parquet
  fato_compras.parquet
  ind_concentracao_item_ano_comprador.parquet
  ind_concentracao_item_ano_uf.parquet
  ind_dispersao_item_ano.parquet
  ind_dominio_item_ano.parquet
  ind_fornecedores_ano.parquet
  ind_item_estado_ano.parquet
  ind_modalidade.parqu

In [15]:
# CONFERÊNCIA de 17/10/2026: resumo das tabelas do painel contra as regras (D54 e seguintes)
hhi06 = pl.read_parquet(SAIDAS['hhi']).select('ano', 'item', 'n_compras_valor')
c = dominio.join(hhi06, on=['ano', 'item'], how='left')
linhas = [
    ('D54 pontos no mapa (esperado 19.798)', dominio.height, dominio.height == 19798),
    ('D54 sem fatia por valor (esperado 700)', int(dominio['fatia_lider_valor'].is_null().sum()), int(dominio['fatia_lider_valor'].is_null().sum()) == 700),
    ('D54 vazia SE E SOMENTE SE < 7 válidas', '', bool((dominio['fatia_lider_valor'].is_null() == (dominio['compras_valor_valido'] < MIN_COMPRAS)).all())),
    ('D54 válidas iguais às da rotina 06', '', bool((c['compras_valor_valido'] == c['n_compras_valor']).all())),
    ('D54 nenhuma fatia por valor igual a zero', '', bool((dominio['fatia_lider_valor'].drop_nulls() > 0).all())),
    ('D29 itens grandes (esperado 4.028)', int(dominio['item_grande'].sum()), int(dominio['item_grande'].sum()) == 4028),
    ('D54 fatia por compras sem vazio', int(dominio['fatia_lider_compras'].is_null().sum()), int(dominio['fatia_lider_compras'].is_null().sum()) == 0),
    ('D54 mediana fatia por valor (esperado ~42,5%)', round(dominio['fatia_lider_valor'].median() * 100, 2), None),   # só informação
]
# D57: as células refazem a ind_modalidade (mesmo % e mesmo número de células nas 7 modalidades)
rep = (ind_mod_cel.group_by('modalidade_grupo').agg(pct(pl.col('dif_log_vs_pregao').median()).alias('r'), pl.len().alias('n'))
         .join(ind_mod.select('modalidade_grupo', 'vs_pregao_mesmo_lote_pct', 'celulas_comparadas'), on='modalidade_grupo'))
linhas += [
    ('D57 células refazem a ind_modalidade (7 modalidades)', rep.height,
     rep.height == 7 and bool((rep['r'] == rep['vs_pregao_mesmo_lote_pct']).all()) and bool((rep['n'] == rep['celulas_comparadas']).all())),
    ('D57 células gravadas (esperado 34.018, teste local de 19/10/2026)', ind_mod_cel.height, ind_mod_cel.height == 34018),
    ('D59 nenhum HHI onde a classe é "Dados insuficientes" (3 tabelas)', '',
     all(t.filter((pl.col(f'classe_{m}') == 'Dados insuficientes') & pl.col(f'hhi_{m}').is_not_null()).height == 0
         for t in [res_hhi_painel, conc_uf, conc_comp] for m in ['compras', 'valor'])),
    ('D59 HHI por compras preenchido (esperado 19.798, D7)', int(res_hhi_painel['hhi_compras'].is_not_null().sum()),
     int(res_hhi_painel['hhi_compras'].is_not_null().sum()) == 19798),
    ('D61 compra_valida = Sim (esperado 802.660, D16)', int((fato_out['compra_valida'] == 'Sim').sum()),
     int((fato_out['compra_valida'] == 'Sim').sum()) == 802660),
    ('D61 entra_preco_relativo = Sim (esperado 747.684, D45; mesmo conjunto da escala, D53)', int((fato_out['entra_preco_relativo'] == 'Sim').sum()),
     int((fato_out['entra_preco_relativo'] == 'Sim').sum()) == 747684),
    ('D62 índice por estado e período: 120 períodos x 23 estados (esperado 2.760)', pl.read_csv(SAIDAS['territorial_periodo']).height,
     pl.read_csv(SAIDAS['territorial_periodo']).height == 2760),
    ('D62 período 2009–2023 igual à res_territorial (23 estados)', '',
     pl.read_csv(SAIDAS['territorial_periodo']).filter((pl.col('ano_inicio') == 2009) & (pl.col('ano_fim') == 2023))
       .select('uf', 'indice', 'registros').sort('uf').equals(pl.read_csv(SAIDAS['territorial']).select('uf', 'indice', 'registros').sort('uf'))),
    ('D59 HHI por valor preenchido (esperado 19.098, D7)', int(res_hhi_painel['hhi_valor'].is_not_null().sum()),
     int(res_hhi_painel['hhi_valor'].is_not_null().sum()) == 19098),
]
for nome, valor, ok in linhas:
    print(f'{"INFO   " if ok is None else "OK     " if ok else "FALHOU "} {nome}: {valor}')
print('Por ano (sem fatia por valor):', dominio.group_by('ano').agg(pl.col('fatia_lider_valor').is_null().sum()).sort('ano').rows())


OK      D54 pontos no mapa (esperado 19.798): 19798
OK      D54 sem fatia por valor (esperado 700): 700
OK      D54 vazia SE E SOMENTE SE < 7 válidas: 
OK      D54 válidas iguais às da rotina 06: 
OK      D54 nenhuma fatia por valor igual a zero: 
OK      D29 itens grandes (esperado 4.028): 4028
OK      D54 fatia por compras sem vazio: 0
INFO    D54 mediana fatia por valor (esperado ~42,5%): 42.46
OK      D57 células refazem a ind_modalidade (7 modalidades): 7
OK      D57 células gravadas (esperado 34.018, teste local de 19/10/2026): 34018
OK      D59 nenhum HHI onde a classe é "Dados insuficientes" (3 tabelas): 
OK      D59 HHI por compras preenchido (esperado 19.798, D7): 19798
OK      D61 compra_valida = Sim (esperado 802.660, D16): 802660
OK      D61 entra_preco_relativo = Sim (esperado 747.684, D45; mesmo conjunto da escala, D53): 747684
OK      D62 índice por estado e período: 120 períodos x 23 estados (esperado 2.760): 2760
OK      D62 período 2009–2023 igual à res_territorial (

In [16]:
# ── CONFERÊNCIA DA RODADA DE 25/10/2026 (D45: índice com 3 compras válidas) ─
# Antes de rodar, copie a pasta Base Painel inteira para "Base Painel anterior" (mesma pasta Base v2).
# Compara a Base Painel nova com a cópia, tabela por tabela, pelo CONTEÚDO (não pelos bytes).
# - a ordem das linhas pode mudar entre execuções (agrupamentos em paralelo): as tabelas são ordenadas antes de comparar;
# - somas em ponto flutuante podem mudar na 16ª casa pela ordem da soma: números decimais aceitam diferença até 1e-9.
# Esperado: todas as tabelas iguais, exceto res_indice e res_robustez, que só ganham as colunas _min3 (rotina 05),
# e rastreabilidade, que só muda o texto da regra do índice. (A rodada de 21/10/2026 conferiu a dim_comprador.)
TOL = 1e-9
def mesma_tabela(a, b):
    if a.columns != b.columns or a.height != b.height: return False, 'colunas ou linhas diferentes'
    a, b = a.sort(a.columns, nulls_last=True), b.sort(b.columns, nulls_last=True)
    for c in a.columns:
        x, y = a[c], b[c]
        if x.dtype in (pl.Float32, pl.Float64):
            if not (x.is_null() == y.is_null()).all(): return False, f'vazios diferentes em {c}'
            d = (x - y).abs().max()
            if d is not None and d > TOL: return False, f'{c} difere em até {d}'
        elif not x.equals(y): return False, f'valores diferentes em {c}'
    return True, ''
COLUNAS_NOVAS = {'res_indice': ['indice_min3', 'itens_no_elo_min3', 'variacao_pct_min3'],   # D45 (compl. 25/10/2026)
                 'res_robustez': ['indice_2023_min3', 'itens_no_maior_elo_min3']}
LINHA_INDICE = 'Índice de preço real 2009-2023'
problemas = []
if not SAIDA_ANTERIOR.exists():
    print('Sem a pasta "Base Painel anterior": conferência não feita.')
for p_novo in (sorted(SAIDA.glob('*.parquet')) if SAIDA_ANTERIOR.exists() else []):
    p_ant = SAIDA_ANTERIOR / p_novo.name
    if not p_ant.exists():
        if p_novo.name == 'res_conc_preco_pares.parquet':  # seção 12B (T27): tabela nova autorizada; validada pelos testes T1/T2/T3 abaixo
            print(f'{p_novo.name:45s} NOVO AUTORIZADO — conferir seção 12B')
            continue
        problemas.append(f'{p_novo.name}: não existe na Base Painel anterior'); continue
    a, b = pl.read_parquet(p_ant), pl.read_parquet(p_novo)
    if p_novo.stem in COLUNAS_NOVAS:
        extra = [x for x in b.columns if x not in a.columns]
        ok, motivo = (False, f'colunas novas: {extra}') if extra != COLUNAS_NOVAS[p_novo.stem] else mesma_tabela(a, b.drop(extra))
        print(f'{p_novo.name:45s} {a.height:>8,} linhas  {"IGUAL + colunas " + ", ".join(extra) if ok else "DIFERENTE: " + motivo}')
        if not ok: problemas.append(f'{p_novo.stem}: {motivo}')
        continue
    if p_novo.stem == 'rastreabilidade':
        regra_nova = b.filter(pl.col('indicador') == LINHA_INDICE)['regra'].to_list()
        regra_ant = a.filter(pl.col('indicador') == LINHA_INDICE)['regra'].to_list()
        b2 = b.with_columns(pl.when(pl.col('indicador') == LINHA_INDICE).then(pl.lit(regra_ant[0] if regra_ant else None))
                              .otherwise(pl.col('regra')).alias('regra'))
        ok, motivo = mesma_tabela(a, b2)
        ok = ok and len(regra_nova) == 1 and 'D45' in regra_nova[0]
        print(f'{p_novo.name:45s} {a.height:>8,} linhas  {"IGUAL, exceto a regra do índice (D45)" if ok else "DIFERENTE: " + (motivo or "regra do índice")}')
        if not ok: problemas.append(f'rastreabilidade: {motivo or "regra do índice sem D45"}')
        continue
    ok, motivo = mesma_tabela(a, b)
    print(f'{p_novo.name:45s} {a.height:>8,} linhas  {"IGUAL" if ok else "DIFERENTE: " + motivo}')
    if not ok: problemas.append(f'{p_novo.name}: {motivo}')
# Se a cópia anterior contiver a tabela 12B, ela será gerada somente na seção seguinte.
faltam = [] if not SAIDA_ANTERIOR.exists() else sorted(
    ({p.name for p in SAIDA_ANTERIOR.glob('*.parquet')} - {p.name for p in SAIDA.glob('*.parquet')})
    - {'res_conc_preco_pares.parquet'}  # seção 12B (T27)
)
if faltam: problemas.append(f'tabelas da Base Painel atual que não foram geradas: {faltam}')
print()
if not SAIDA_ANTERIOR.exists():
    pass
elif problemas:
    print('Diferenças encontradas:'); [print('-', x) for x in problemas]
else:
    print('CONFERE: só mudaram res_indice, res_robustez (colunas _min3) e a regra do índice na rastreabilidade (D45).')


## 12B. Concentração × preço — pares de anos consecutivos (T27; 24/10/2026)

**Complemento da rotina 12, sem alteração dos indicadores anteriores.** Esta seção incorpora a rotina independente `12B_CONC_PRECO_PARES.py` após as conferências da seção 11.

- **Entradas:** `Base Painel/fato_compras.parquet`, `Base Painel/res_hhi.parquet` (já produzidos nesta rotina) e `Base Correlacao - Publico/correlacao_desenho_b.csv` e `correlacao_testes_b.csv` (rotina 09).
- **Cálculo:** pares de anos **consecutivos** por item; primeira diferença do HHI (`d_hhi`) e variação relativa do preço real mediano (`d_preco`), separadamente por compras e por valor, com quintis fixos da diferença de HHI. Mantém D7, D16, D31 e D34 da rotina 09.
- **Saídas adicionais:** `Base Painel/res_conc_preco_pares.parquet` e `.csv`, sem modificar as demais tabelas do painel.
- **Conferências obrigatórias antes de gravar:** **T1** quantidades de pares iguais à rotina 09 (referência: 13.586 por compras e 13.088 por valor), **T2** 10 medianas e respectivas contagens por quintil iguais à rotina 09. **T3:** mesmos SHA-256 dos demais arquivos da Base Painel antes e depois.

A conferência D24 da seção 11 foi ajustada **somente** para reconhecer `res_conc_preco_pares.parquet` como inclusão autorizada. Execute a seção 12B depois da seção 11; se um teste falhar, nenhuma saída 12B deverá ser considerada aprovada. Os arquivos 12B existentes podem ser atualizados na reexecução, como no script original. **Esta cópia ainda não foi executada com os dados reais.**


In [17]:
# SEÇÃO 12B — Concentração × preço (T27; 24/10/2026)
# Adaptada de 12B_CONC_PRECO_PARES.py: mesmos cálculos, arquivos e testes T1/T2/T3.
# Encapsulada em função para NÃO sobrescrever variáveis usadas pelas seções 1 a 11.
# Somente res_conc_preco_pares.parquet/.csv são criados ou atualizados.
# Em caso de erro, a execução interrompe esta célula (sem sys.exit do Colab).
def executar_secao_12b():
    import os, hashlib
    from pathlib import Path
    import polars as pl

    PASTA_DADOS = Path(globals()['PASTA_DADOS'])  # mesmo caminho da rotina 12
    BP = Path(globals()['SAIDA'])  # mesma Base Painel da rotina 12
    ARQ_FATO, ARQ_HHI = BP / 'fato_compras.parquet', BP / 'res_hhi.parquet'
    ARQ_B = PASTA_DADOS / 'Base Correlacao - Publico' / 'correlacao_desenho_b.csv'
    ARQ_TB = PASTA_DADOS / 'Base Correlacao - Publico' / 'correlacao_testes_b.csv'
    SAIDA = 'res_conc_preco_pares'
    MIN_COMPRAS = globals()['MIN_COMPRAS']              # D7: reutiliza exatamente o parâmetro da rotina 12
    ROTULOS = ['Q1 queda forte', 'Q2', 'Q3', 'Q4', 'Q5 alta forte']   # D31, mesmos rótulos da rotina 09

    def sair(cod, msg):
        raise RuntimeError(f'Seção 12B interrompida (código {cod}): {msg}')

    for a in [ARQ_FATO, ARQ_HHI, ARQ_B, ARQ_TB]:
        if not a.exists(): sair(2, f'NÃO ENCONTRADO: {a}')

    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for bloco in iter(lambda: f.read(1 << 20), b''): h.update(bloco)
        return h.hexdigest()
    antes = {p.name: sha(p) for p in sorted(BP.iterdir()) if p.is_file() and not p.name.startswith(SAIDA)}
    print(f'Base Painel: {len(antes)} arquivos existentes registrados (SHA-256)')

    # preço do item no ano = mediana do preço real das compras válidas, a mesma coluna que a rotina 09 usa (rotina 04B)
    fato = pl.read_parquet(ARQ_FATO, columns=['ano', 'item', 'mediana_item_ano', 'compra_valida'])
    preco = (fato.filter(pl.col('compra_valida') == 'Sim').group_by(['ano', 'item'])
                 .agg(pl.col('mediana_item_ano').first().alias('preco')))
    hhi = pl.read_parquet(ARQ_HHI)
    MEDIDAS = {'compras': ('hhi_compras', 'n_compras'), 'valor': ('hhi_valor', 'n_compras_valor')}
    partes = []
    for m, (col_h, col_n) in MEDIDAS.items():
        pm = (hhi.filter(pl.col(col_n) >= MIN_COMPRAS).select(['ano', 'item', pl.col(col_h).alias('hhi')])
                 .join(preco, on=['ano', 'item'], how='inner').filter(pl.col('preco') > 0))
        dd = (pm.sort(['item', 'ano'])
                .with_columns(pl.col('ano').shift(1).over('item').alias('ano_anterior'),
                              pl.col('hhi').shift(1).over('item').alias('hhi_anterior'),
                              pl.col('preco').shift(1).over('item').alias('preco_anterior'))
                .with_columns((pl.col('hhi') - pl.col('hhi_anterior')).alias('d_hhi'),
                              (pl.col('preco') / pl.col('preco_anterior') - 1).alias('d_preco'))
                .filter(((pl.col('ano') - pl.col('ano_anterior')) == 1) & pl.col('d_hhi').is_not_null() & pl.col('d_preco').is_not_null()))
        # quintil da mudança da concentração, calculado sobre TODOS os pares da leitura (fixo; o filtro do painel só escolhe pares)
        dd = dd.with_columns(pl.col('d_hhi').qcut(5, labels=ROTULOS, allow_duplicates=True).cast(pl.Utf8).alias('quintil'),
                             pl.lit(m).alias('medida'))
        partes.append(dd)
    pares = (pl.concat(partes)
               .with_columns(pl.col('quintil').replace_strict({r: i + 1 for i, r in enumerate(ROTULOS)}, return_dtype=pl.Int8).alias('ordem_quintil'))
               .select(['medida', 'item', 'ano', 'ano_anterior', 'hhi_anterior', 'hhi', 'd_hhi',
                        'preco_anterior', 'preco', 'd_preco', 'quintil', 'ordem_quintil'])
               .sort(['medida', 'item', 'ano']))

    # ---------- testes ----------
    falhas = []
    tb = pl.read_csv(ARQ_TB)
    for m in MEDIDAS:
        n_aqui = pares.filter(pl.col('medida') == m).height
        n_09 = tb.filter(pl.col('medida') == m)['pares'][0]
        print(f'T1 [{m}] pares: {n_aqui:,} | rotina 09: {n_09:,}')
        if n_aqui != n_09: falhas.append(f'T1 {m}: {n_aqui} ≠ {n_09}')
    refeito = (pares.group_by(['medida', 'quintil']).agg((pl.col('d_preco').median() * 100).round(2).alias('aqui'), pl.len().alias('n_aqui')))
    b = pl.read_csv(ARQ_B).join(refeito, on=['medida', 'quintil'], how='left').sort(['medida', 'quintil'])
    print(b)
    dif = b.filter((pl.col('aqui') != pl.col('var_preco_mediana_pct')) | (pl.col('n_aqui') != pl.col('n')))
    if b['aqui'].null_count() > 0 or dif.height > 0: falhas.append(f'T2: {dif.height} quintis diferentes da rotina 09')
    else: print('T2 ok: as 10 medianas e os 10 números de pares batem com correlacao_desenho_b')

    if falhas: sair(1, 'FALHOU — nada foi gravado:\n  ' + '\n  '.join(falhas))

    # ---------- gravar (só arquivos novos) e T3 ----------
    tmp_parq, tmp_csv = BP / f'{SAIDA}.parquet.tmp', BP / f'{SAIDA}.csv.tmp'
    pares.write_parquet(tmp_parq); pares.write_csv(tmp_csv)
    os.replace(tmp_parq, BP / f'{SAIDA}.parquet'); os.replace(tmp_csv, BP / f'{SAIDA}.csv')
    depois = {p.name: sha(p) for p in sorted(BP.iterdir()) if p.is_file() and not p.name.startswith(SAIDA)}
    mudou = [n for n in antes if depois.get(n) != antes[n]] + [n for n in depois if n not in antes]
    if mudou:
        for n in [f'{SAIDA}.parquet', f'{SAIDA}.csv']: (BP / n).unlink(missing_ok=True)
        sair(1, f'T3 FALHOU — arquivos existentes mudaram ou apareceram: {mudou}. Os arquivos novos foram apagados.')
    print(f'T3 ok: os {len(antes)} arquivos que já existiam continuam com o mesmo SHA-256')
    print(f'Gravado: Base Painel/{SAIDA}.parquet e .csv ({pares.height:,} linhas; compras {partes[0].height:,}, valor {partes[1].height:,})')
    return pares.height  # execução concluída; não encerra o notebook

linhas_12b = executar_secao_12b()


Base Painel: 47 arquivos existentes registrados (SHA-256)
T1 [compras] pares: 13,586 | rotina 09: 13,586
T1 [valor] pares: 13,088 | rotina 09: 13,088
shape: (10, 6)
┌─────────┬────────────────┬───────────────────────┬──────┬───────┬────────┐
│ medida  ┆ quintil        ┆ var_preco_mediana_pct ┆ n    ┆ aqui  ┆ n_aqui │
│ ---     ┆ ---            ┆ ---                   ┆ ---  ┆ ---   ┆ ---    │
│ str     ┆ str            ┆ f64                   ┆ i64  ┆ f64   ┆ u32    │
╞═════════╪════════════════╪═══════════════════════╪══════╪═══════╪════════╡
│ compras ┆ Q1 queda forte ┆ -0.94                 ┆ 2722 ┆ -0.94 ┆ 2722   │
│ compras ┆ Q2             ┆ -1.49                 ┆ 2713 ┆ -1.49 ┆ 2713   │
│ compras ┆ Q3             ┆ -1.57                 ┆ 2717 ┆ -1.57 ┆ 2717   │
│ compras ┆ Q4             ┆ -2.02                 ┆ 2717 ┆ -2.02 ┆ 2717   │
│ compras ┆ Q5 alta forte  ┆ -2.67                 ┆ 2717 ┆ -2.67 ┆ 2717   │
│ valor   ┆ Q1 queda forte ┆ -1.47                 ┆ 2618 ┆ -1.47